In [4]:
import torch
from test_attn_order_eval import AttnOrderEval   # signal-agnostic ranking eval over a [T, L] per-step score + order

class MarginOrderEval(AttnOrderEval):

    def __init__(self, margin, order):                              # margin [T, L] (already preprocessed, p1 - p2), order [T] long
        assert margin.dim() == 2, "margin.dim(): {} == 2 false".format(margin.dim())

        super().__init__(margin, order)                             # reuse geometry + recall_at_h / pr_auc / ndcg_at_h
    # end

    def get_margin(self):
        return self.attn   # the base stores the ranking score here (generic [T, L] score slot)
    # end
# end


class ConfidenceOrderEval(AttnOrderEval):

    def __init__(self, confidence, order):                          # confidence [T, L] (already preprocessed), order [T] long
        assert confidence.dim() == 2, "confidence.dim(): {} == 2 false".format(confidence.dim())

        super().__init__(confidence, order)                         # reuse geometry + recall_at_h / pr_auc / ndcg_at_h
    # end

    def get_confidence(self):
        return self.attn   # the base stores the ranking score here (generic [T, L] score slot)
    # end
# end


def summ(x):
    return "{:.3f} (n={})".format(x.nanmean().item(), int((~x.isnan()).sum()))
# end

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(0)
    T = L = 64

    num_batch = 6
    num_blk = 4

    for id_batch in range(num_batch):

        folder_base = f'stats/{id_batch}'
        pos_root = 32

        for id_blk in range(num_blk):

            pos_base = pos_root + id_blk * L
            unmask = torch.load(f'{folder_base}/unmask_{pos_base}_{pos_base + L}.pt')
            unmask = unmask.squeeze(-1) - pos_base
            order = unmask

            step_of = torch.full((L,), -1, dtype=torch.long)
            step_of[order] = torch.arange(T)
            gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

            # -------- margin --------
            margin_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))   # larger margin = sooner
            margin_random = torch.rand(T, L, dtype=torch.float64)
            margin = torch.load(f'{folder_base}/margin_{pos_base}_{pos_base + L}.pt')

            print(f'\n\n\n-------- margin --------')
            for name, margin in [("perfect", margin_perfect), ("random ", margin_random), (f"margin {id_batch} {pos_base}", margin)]:
                ev = MarginOrderEval(margin, order)
                print("{}  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
                    name, summ(ev.recall_at_h(5)), summ(ev.pr_auc(5)), summ(ev.ndcg_at_h(10))))
            # end


            # -------- conf --------
            print(f'\n-------- conf --------')
            conf_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))   # higher conf = sooner
            conf_random = torch.rand(T, L, dtype=torch.float64)
            conf = torch.load(f'{folder_base}/conf_{pos_base}_{pos_base + L}.pt')

            for name, conf in [("perfect", conf_perfect), ("random ", conf_random), (f"conf {id_batch} {pos_base}", conf)]:
                ev = ConfidenceOrderEval(conf, order)
                print("{}  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
                    name, summ(ev.recall_at_h(5)), summ(ev.pr_auc(5)), summ(ev.ndcg_at_h(10))))
            # end


            # -------- attn --------
            print(f'\n-------- attn --------')    
            attn_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))
            attn_random = torch.rand(T, L, dtype=torch.float64)

            attn = torch.load(f'{folder_base}/attn_{pos_base}_{pos_base + L}.pt')
            attn_last = attn[:,-1,:,:].squeeze(1) # -> [step, attn]
            attn_last_switched = attn_last[unmask, :]

            for name, attn in [("perfect", attn_perfect), ("random ", attn_random), (f"attn {id_batch} {pos_base}", attn_last_switched)]:
                ev = AttnOrderEval(attn, order)
                r = ev.recall_at_h(5).nanmean().item()
                p = ev.pr_auc(5).nanmean().item()
                nd = ev.ndcg_at_h(10).nanmean().item()
                print("{}  recall@5={:.3f}  pr_auc@5={:.3f}  ndcg@10={:.3f}".format(name, r, p, nd))
            # end
        # end
    # end
# end




-------- margin 0 32 --------
perfect  recall@5=1.000 (n=59)  pr_auc@5=1.000 (n=58)  ndcg@10=1.000 (n=62)
random   recall@5=0.275 (n=59)  pr_auc@5=0.319 (n=58)  ndcg@10=0.412 (n=62)
margin  recall@5=0.600 (n=59)  pr_auc@5=0.687 (n=58)  ndcg@10=0.759 (n=62)

-------- conf 0 32 --------
perfect  recall@5=1.000 (n=59)  pr_auc@5=1.000 (n=58)  ndcg@10=1.000 (n=62)
random   recall@5=0.254 (n=59)  pr_auc@5=0.324 (n=58)  ndcg@10=0.413 (n=62)
conf  recall@5=0.766 (n=59)  pr_auc@5=0.829 (n=58)  ndcg@10=0.869 (n=62)

-------- attn 0 32 --------
perfect  recall@5=1.000  pr_auc@5=1.000  ndcg@10=1.000
random   recall@5=0.227  pr_auc@5=0.293  ndcg@10=0.382
attn  recall@5=0.671  pr_auc@5=0.757  ndcg@10=0.806



-------- margin 0 96 --------
perfect  recall@5=1.000 (n=59)  pr_auc@5=1.000 (n=58)  ndcg@10=1.000 (n=62)
random   recall@5=0.231 (n=59)  pr_auc@5=0.288 (n=58)  ndcg@10=0.376 (n=62)
margin  recall@5=0.603 (n=59)  pr_auc@5=0.685 (n=58)  ndcg@10=0.760 (n=62)

-------- conf 0 96 --------
perfec